# FMCW training
Train a separate run with the paper architectures and fixed partitions. Training is disabled until explicitly enabled.


In [ ]:
from pathlib import Path
import sys
try:
    from google.colab import drive
except ImportError:
    PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'training/fmcw_training').is_dir())
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/fmcw-mitigation-detection')
sys.path.insert(0, str(PROJECT/'training'))
print(PROJECT)
# Install dependencies once if needed:
# %pip install -r {PROJECT/'training/requirements.txt'}


## Settings
Select `full`, `unconditioned` or `unet_iq`. The training seed never changes the data partitions.


In [ ]:
from protocol import entry
MODEL = 'full'
SEED = entry(MODEL)['seed']  # May also be 12,26 or 27 for any model.
EPOCHS = 120
DEVICE = 'auto'
RUN_DIR = PROJECT/'training/runs'/f'{MODEL}_seed{SEED}'
RUN_TRAINING = False


## Fixed partitions
Load the original dataset archives. This step reads catalogs, without test waveform inference.


In [ ]:
from protocol import load_data
main, external, splits = load_data()
print({name:len(indices) for name,indices in splits.items()})


## Train or resume
All 120epochs use the complete training and validation partitions. Selection is maximum validation F1 at 0.5, with the earliest exact tie. No test evaluation or calibration occurs here.


In [ ]:
from train import run
if RUN_TRAINING:
    result = run(MODEL, SEED, EPOCHS, RUN_DIR, DEVICE)
    print(result)
else:
    print('Training disabled. Set RUN_TRAINING=True when ready.')


The selected export is `RUN_DIR/selected.pt`; full epoch checkpoints remain in `RUN_DIR/complet/`. Use `infer.py --weights` to inspect it. Run `summarize.py` to plot the supplied nine-run histories.
